# Candidate Preprocessing Pipeline
Complete end-to-end pipeline including EDA, validation, normalization, embeddings, and hybrid retrieval.

In [1]:
from __future__ import annotations

import json
import re
from collections import Counter
from pathlib import Path
from typing import Any
import numpy as np

import pandas as pd
from sklearn.preprocessing import MinMaxScaler

INPUT_PATH = Path("candidates.jsonl")
OUTPUT_PATH = Path("top_10k_candidates.parquet")
CSV_OUTPUT_PATH = Path("top_10k_candidates.csv")
TOP_N = 10000

SKILL_WEIGHT = {"beginner": 1.0, "intermediate": 2.0, "advanced": 3.0, "expert": 4.0}


def safe_get(record: dict[str, Any], *keys: str, default: Any = None) -> Any:
    current: Any = record
    for key in keys:
        if not isinstance(current, dict):
            return default
        current = current.get(key, default)
    return current if current is not None else default


def as_text(value: Any) -> str:
    if value is None:
        return ""
    if isinstance(value, str):
        return value.strip()
    return str(value).strip()


def normalize_list(values: Any) -> list[Any]:
    return values if isinstance(values, list) else []


def join_nonempty(values: list[str], separator: str = " | ") -> str:
    return separator.join([value for value in values if value])


def parse_jsonl(path: Path) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f"Invalid JSON on line {line_number}: {exc}") from exc
    return rows


def flatten_skills(record: dict[str, Any]) -> tuple[str, float, list[str]]:
    skills = normalize_list(record.get("skills"))
    names: list[str] = []
    weighted_score = 0.0
    for skill in skills:
        if not isinstance(skill, dict):
            continue
        name = as_text(skill.get("name"))
        if not name:
            continue
        names.append(name)
        proficiency = as_text(skill.get("proficiency")).lower()
        endorsements = skill.get("endorsements", 0) or 0
        duration_months = skill.get("duration_months", 0) or 0
        weighted_score += SKILL_WEIGHT.get(proficiency, 0.5) + min(float(endorsements), 25.0) / 25.0 + min(float(duration_months), 60.0) / 60.0
    return ", ".join(names), weighted_score, names


def flatten_education(record: dict[str, Any]) -> str:
    education = normalize_list(record.get("education"))
    parts: list[str] = []
    for item in education:
        if not isinstance(item, dict):
            continue
        fields = [
            as_text(item.get("degree")),
            as_text(item.get("field_of_study")),
            as_text(item.get("institution")),
            f"{item.get('start_year', '')}-{item.get('end_year', '')}".strip("-"),
        ]
        entry = join_nonempty([field for field in fields if field and field != "-"])
        if entry:
            parts.append(entry)
    return join_nonempty(parts, " || ")


def flatten_career_history(record: dict[str, Any]) -> str:
    career_history = normalize_list(record.get("career_history"))
    parts: list[str] = []
    for item in career_history:
        if not isinstance(item, dict):
            continue
        title = as_text(item.get("title"))
        company = as_text(item.get("company"))
        industry = as_text(item.get("industry"))
        description = as_text(item.get("description"))
        summary = join_nonempty([title, company, industry, description])
        if summary:
            parts.append(summary)
    return join_nonempty(parts, " || ")


def flatten_redrob_signals(record: dict[str, Any]) -> dict[str, Any]:
    signals = record.get("redrob_signals") if isinstance(record.get("redrob_signals"), dict) else {}
    return {
        "profile_completeness_score": signals.get("profile_completeness_score"),
        "signup_date": signals.get("signup_date"),
        "last_active_date": signals.get("last_active_date"),
        "open_to_work_flag": signals.get("open_to_work_flag"),
        "profile_views_received_30d": signals.get("profile_views_received_30d"),
        "applications_submitted_30d": signals.get("applications_submitted_30d"),
        "recruiter_response_rate": signals.get("recruiter_response_rate"),
        "avg_response_time_hours": signals.get("avg_response_time_hours"),
        "connection_count": signals.get("connection_count"),
        "endorsements_received": signals.get("endorsements_received"),
        "notice_period_days": signals.get("notice_period_days"),
    }


def compute_semantic_similarity(profile_text: str, skills_text: str, history_text: str, education_text: str) -> float:
    parts = [profile_text, skills_text, history_text, education_text]
    token_counts: Counter[str] = Counter()
    for part in parts:
        token_counts.update(re.findall(r"[a-z0-9+.#-]+", part.lower()))
    if not token_counts:
        return 0.0
    important_tokens = {
        "python", "sql", "spark", "airflow", "ml", "machine", "learning", "llm", "nlp", "opencv",
        "tensorflow", "pytorch", "kubernetes", "docker", "aws", "gcp", "azure", "react", "node",
        "fastapi", "flask", "databricks", "snowflake", "faiss", "vector", "search",
    }
    weighted = sum(token_counts[token] for token in token_counts if token in important_tokens)
    raw = weighted / max(len(token_counts), 1)
    return round(min(1.0, raw / 4.0), 4)


def compute_retrieval_score(record: dict[str, Any], skill_weighted_score: float) -> float:
    profile = record.get("profile") if isinstance(record.get("profile"), dict) else {}
    redrob = record.get("redrob_signals") if isinstance(record.get("redrob_signals"), dict) else {}
    years_experience = profile.get("years_of_experience")
    years_experience = float(years_experience) if isinstance(years_experience, (int, float)) else 0.0
    completeness = float(redrob.get("profile_completeness_score") or 0.0)
    activity = 0.0
    if redrob.get("open_to_work_flag"):
        activity += 5.0
    activity += min(float(redrob.get("profile_views_received_30d") or 0.0), 200.0) / 20.0
    activity += min(float(redrob.get("applications_submitted_30d") or 0.0), 20.0) / 10.0
    activity += min(float(redrob.get("recruiter_response_rate") or 0.0), 1.0) * 10.0
    score = (
        completeness * 0.25
        + years_experience * 2.0
        + skill_weighted_score * 4.0
        + activity
        + min(float(redrob.get("endorsements_received") or 0.0), 200.0) / 8.0
    )
    return round(score, 4)


print("✓ Loading candidates data...")
records = parse_jsonl(INPUT_PATH)
print(f"✓ Loaded {len(records)} records")

rows: list[dict[str, Any]] = []

for record in records:
    profile = record.get("profile") if isinstance(record.get("profile"), dict) else {}
    skills_text, skill_weighted_score, skill_names = flatten_skills(record)
    education_text = flatten_education(record)
    career_history_text = flatten_career_history(record)
    redrob_signals = flatten_redrob_signals(record)

    candidate_document_parts = [
        as_text(profile.get("headline")),
        as_text(profile.get("summary")),
        skills_text,
        career_history_text,
        education_text,
        as_text(record.get("profile", {}).get("location")),
        as_text(record.get("profile", {}).get("current_title")),
        as_text(record.get("profile", {}).get("current_company")),
    ]

    candidate_document = join_nonempty(candidate_document_parts, "\n")
    semantic_similarity = compute_semantic_similarity(
        as_text(profile.get("summary")),
        skills_text,
        career_history_text,
        education_text,
    )
    retrieval_score = compute_retrieval_score(record, skill_weighted_score)

    candidate_id = as_text(record.get("candidate_id")) or f"CAND_{len(rows) + 1:07d}"

    rows.append(
        {
            "candidate_id": candidate_id,
            "candidate_document": candidate_document,
            "semantic_similarity": semantic_similarity,
            "retrieval_score": retrieval_score,
            "skill_match": round(skill_weighted_score, 4),
            "skills": skill_names,
            "career_history": normalize_list(record.get("career_history")),
            "profile": profile,
            "education": normalize_list(record.get("education")),
            "redrob_signals": redrob_signals,
            "years_experience": profile.get("years_of_experience"),
        }
    )

processed = pd.DataFrame(rows)
print(f"✓ Created {len(processed)} candidate records")
print(f"\nDataFrame shape: {processed.shape}")

✓ Loading candidates data...
✓ Loaded 100000 records
✓ Created 100000 candidate records

DataFrame shape: (100000, 11)


## 1. Exploratory Data Analysis (EDA)

In [2]:
print("="*80)
print("EXPLORATORY DATA ANALYSIS (EDA)")
print("="*80)

print(f"\nDataFrame Info:")
print(f"  Total Records: {len(processed)}")
print(f"  Total Columns: {len(processed.columns)}")
print(f"  Memory Usage: {processed.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print(f"\nColumn Data Types:")
print(processed.dtypes)

print(f"\nMissing Values:")
missing = processed.isna().sum()
print(missing[missing > 0] if missing.any() else "  No missing values")

print(f"\nNumeric Statistics:")
numeric_cols = ["semantic_similarity", "retrieval_score", "skill_match", "years_experience"]
for col in numeric_cols:
    if col in processed.columns:
        valid = processed[col].dropna()
        print(f"  {col}:")
        print(f"    Count: {len(valid)}, Mean: {valid.mean():.4f}, Std: {valid.std():.4f}")
        print(f"    Min: {valid.min():.4f}, Max: {valid.max():.4f}")

print(f"\nCandidate Document Length Statistics:")
doc_lengths = processed["candidate_document"].str.len()
print(f"  Mean: {doc_lengths.mean():.0f} chars, Max: {doc_lengths.max():.0f}, Min: {doc_lengths.min():.0f}")

print(f"\nSkill Distribution (top 20 skills):")
all_skills = [skill for skills_list in processed["skills"] for skill in skills_list]
from collections import Counter
skill_counts = Counter(all_skills)
for skill, count in skill_counts.most_common(20):
    print(f"  {skill}: {count}")

print(f"\nEDA Complete ✓")

EXPLORATORY DATA ANALYSIS (EDA)

DataFrame Info:
  Total Records: 100000
  Total Columns: 11
  Memory Usage: 315.04 MB

Column Data Types:
candidate_id               str
candidate_document         str
semantic_similarity    float64
retrieval_score        float64
skill_match            float64
skills                  object
career_history          object
profile                 object
education               object
redrob_signals          object
years_experience       float64
dtype: object

Missing Values:
  No missing values

Numeric Statistics:
  semantic_similarity:
    Count: 100000, Mean: 0.0065, Std: 0.0057
    Min: 0.0000, Max: 0.0483
  retrieval_score:
    Count: 100000, Mean: 131.5611, Std: 42.6366
    Min: 48.2467, Max: 489.9867
  skill_match:
    Count: 100000, Mean: 22.5530, Std: 10.2201
    Min: 6.1800, Max: 105.4467
  years_experience:
    Count: 100000, Mean: 7.1663, Std: 3.8246
    Min: 1.0000, Max: 16.9000

Candidate Document Length Statistics:
  Mean: 2120 chars, Max: 

## 2. Data Validation

In [3]:
print("="*80)
print("DATA VALIDATION")
print("="*80)

def validate_candidates(df: pd.DataFrame) -> dict[str, Any]:
    """Validate candidate data quality"""
    issues = {
        "empty_candidate_ids": [],
        "empty_documents": [],
        "invalid_scores": [],
        "duplicates": [],
        "validation_passed": True
    }
    
    # Check candidate IDs
    empty_ids = df[df["candidate_id"].fillna("").str.strip() == ""].index.tolist()
    if empty_ids:
        issues["empty_candidate_ids"] = empty_ids
        issues["validation_passed"] = False
    
    # Check documents
    empty_docs = df[df["candidate_document"].fillna("").str.strip() == ""].index.tolist()
    if empty_docs:
        issues["empty_documents"] = empty_docs
        issues["validation_passed"] = False
    
    # Check score ranges
    invalid_scores = df[
        (df["semantic_similarity"] < 0) | (df["semantic_similarity"] > 1) |
        (df["retrieval_score"] < 0) |
        (df["skill_match"] < 0)
    ].index.tolist()
    if invalid_scores:
        issues["invalid_scores"] = invalid_scores
        issues["validation_passed"] = False
    
    # Check duplicates
    duplicates = df[df.duplicated(subset=["candidate_id"], keep=False)].index.tolist()
    if duplicates:
        issues["duplicates"] = duplicates
        issues["validation_passed"] = False
    
    return issues

validation_results = validate_candidates(processed)

print(f"✓ Empty Candidate IDs: {len(validation_results['empty_candidate_ids'])}")
print(f"✓ Empty Documents: {len(validation_results['empty_documents'])}")
print(f"✓ Invalid Scores: {len(validation_results['invalid_scores'])}")
print(f"✓ Duplicate IDs: {len(validation_results['duplicates'])}")
print(f"\nValidation Status: {'PASSED ✓' if validation_results['validation_passed'] else 'FAILED ✗'}")

DATA VALIDATION
✓ Empty Candidate IDs: 0
✓ Empty Documents: 0
✓ Invalid Scores: 0
✓ Duplicate IDs: 0

Validation Status: PASSED ✓


## 3. Normalization & Score Computation

In [4]:
print("="*80)
print("NORMALIZATION & SCORE COMPUTATION")
print("="*80)

def normalize_output_frame(df: pd.DataFrame) -> pd.DataFrame:
    """Normalize and clean output dataframe"""
    normalized = df.copy()
    
    # Clean candidate IDs
    normalized["candidate_id"] = normalized["candidate_id"].fillna("").astype(str).str.strip()
    
    # Clean documents
    normalized["candidate_document"] = normalized["candidate_document"].fillna("").astype(str)
    
    # Normalize scores to 0-1 range
    normalized["semantic_similarity"] = pd.to_numeric(normalized["semantic_similarity"], errors="coerce").fillna(0.0)
    normalized["semantic_similarity"] = normalized["semantic_similarity"].clip(0, 1)
    
    normalized["retrieval_score"] = pd.to_numeric(normalized["retrieval_score"], errors="coerce").fillna(0.0)
    normalized["skill_match"] = pd.to_numeric(normalized["skill_match"], errors="coerce").fillna(0.0)
    
    # Normalize skill_match to 0-1 range
    skill_match_max = normalized["skill_match"].max()
    if skill_match_max > 1:
        normalized["skill_match"] = normalized["skill_match"] / skill_match_max
    normalized["skill_match"] = normalized["skill_match"].clip(0, 1)
    
    # Normalize retrieval_score to 0-1 range
    retrieval_max = normalized["retrieval_score"].max()
    if retrieval_max > 1:
        normalized["retrieval_score"] = normalized["retrieval_score"] / retrieval_max
    normalized["retrieval_score"] = normalized["retrieval_score"].clip(0, 1)
    
    normalized["years_experience"] = pd.to_numeric(normalized["years_experience"], errors="coerce")
    
    # Remove empty candidates
    normalized = normalized[normalized["candidate_id"] != ""].copy()
    
    # Remove duplicates
    normalized = normalized.drop_duplicates(subset=["candidate_id"], keep="first").reset_index(drop=True)
    
    # Sort by quality metrics
    normalized = normalized.sort_values(
        ["retrieval_score", "semantic_similarity", "skill_match"],
        ascending=[False, False, False]
    ).reset_index(drop=True)
    
    # Keep top N
    normalized = normalized.head(TOP_N).reset_index(drop=True)
    
    return normalized

processed = normalize_output_frame(processed)
print(f"✓ Normalized {len(processed)} records")
print(f"\nScore Ranges After Normalization:")
print(f"  Semantic Similarity: [{processed['semantic_similarity'].min():.4f}, {processed['semantic_similarity'].max():.4f}]")
print(f"  Skill Match: [{processed['skill_match'].min():.4f}, {processed['skill_match'].max():.4f}]")
print(f"  Retrieval Score: [{processed['retrieval_score'].min():.4f}, {processed['retrieval_score'].max():.4f}]")

NORMALIZATION & SCORE COMPUTATION
✓ Normalized 10000 records

Score Ranges After Normalization:
  Semantic Similarity: [0.0011, 0.0466]
  Skill Match: [0.3047, 1.0000]
  Retrieval Score: [0.3941, 1.0000]


## 4. Embedding Generation

In [5]:
print("="*80)
print("EMBEDDING GENERATION & SEMANTIC SIMILARITY")
print("="*80)

try:
    from sentence_transformers import SentenceTransformer
    print("Loading sentence transformer model...")
    model = SentenceTransformer("all-MiniLM-L6-v2")
    use_embeddings = True
    print(f"✓ Model loaded: {model.get_sentence_embedding_dimension()} dimensions")
except ImportError:
    print("⚠ sentence-transformers not installed, using keyword-based similarity")
    use_embeddings = False

def compute_semantic_similarity_embedding(text: str, model=None) -> float:
    """Compute semantic similarity using embeddings"""
    if not use_embeddings or model is None:
        return 0.0
    
    if not text or len(text.strip()) == 0:
        return 0.0
    
    try:
        embedding = model.encode(text[:512], convert_to_numpy=True)
        return float(np.mean(np.abs(embedding)))
    except:
        return 0.0

# Update semantic similarity scores using embeddings
print(f"\nProcessing {len(processed)} documents...")
if use_embeddings:
    embeddings_list = []
    batch_size = 32
    for i in range(0, len(processed), batch_size):
        batch_texts = processed.iloc[i:i+batch_size]["candidate_document"].tolist()
        batch_embeddings = model.encode(batch_texts, convert_to_tensor=False, show_progress_bar=(i % 128 == 0))
        embeddings_list.extend(batch_embeddings)
        if (i + batch_size) % 128 == 0:
            print(f"  Processed {min(i + batch_size, len(processed))}/{len(processed)} documents")
    
    # Normalize embeddings to 0-1 range for semantic_similarity
    embeddings_array = np.array(embeddings_list)
    embedding_norms = np.linalg.norm(embeddings_array, axis=1, keepdims=True)
    embedding_norms = np.maximum(embedding_norms, 1e-10)
    normalized_embeddings = embeddings_array / embedding_norms
    
    # Use the mean absolute value of embedding as proxy for semantic_similarity
    processed["semantic_similarity"] = np.mean(np.abs(normalized_embeddings), axis=1)
    processed["semantic_similarity"] = (processed["semantic_similarity"] - processed["semantic_similarity"].min()) / (processed["semantic_similarity"].max() - processed["semantic_similarity"].min() + 1e-10)
    processed["semantic_similarity"] = processed["semantic_similarity"].round(4)
    
    print(f"✓ Embeddings generated")
else:
    print(f"✓ Using keyword-based similarity (no embeddings)")

print(f"\nSemantic Similarity Statistics:")
print(f"  Mean: {processed['semantic_similarity'].mean():.4f}")
print(f"  Std: {processed['semantic_similarity'].std():.4f}")
print(f"  Min: {processed['semantic_similarity'].min():.4f}")
print(f"  Max: {processed['semantic_similarity'].max():.4f}")

EMBEDDING GENERATION & SEMANTIC SIMILARITY


c:\Users\destr\MYprojects\hackathon\JobFilter\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading sentence transformer model...


c:\Users\destr\MYprojects\hackathon\JobFilter\venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\destr\.cache\huggingface\hub\models--sentence-transformers--all-MiNiLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 639.70it/s]
C:\Users\de

✓ Model loaded: 384 dimensions

Processing 10000 documents...


...

  Processed 9856/10000 documents


Batches: 100%|██████████| 1/1 [00:03<00:00,  3.55s/it]


  Processed 9984/10000 documents


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.72s/it]


✓ Embeddings generated

Semantic Similarity Statistics:
  Mean: 0.4919
  Std: 0.1266
  Min: 0.0000
  Max: 1.0000


## 5. Hybrid Retrieval Score

In [6]:
print("="*80)
print("HYBRID RETRIEVAL SCORE COMPUTATION")
print("="*80)

def compute_hybrid_retrieval_score(row: pd.Series) -> float:
    """
    Compute hybrid retrieval score combining:
    - Semantic similarity (embedding-based)
    - Skill match (keyword-based)
    - Profile quality (years of experience, profile completeness)
    """
    semantic_weight = 0.40
    skill_weight = 0.35
    quality_weight = 0.25
    
    semantic_sim = float(row.get("semantic_similarity", 0.0))
    skill_match = float(row.get("skill_match", 0.0))
    
    # Quality score from years of experience and engagement signals
    years_exp = float(row.get("years_experience", 0.0))
    quality_score = min(years_exp / 20.0, 1.0)  # Normalize to 0-1
    
    hybrid_score = (
        semantic_weight * semantic_sim +
        skill_weight * skill_match +
        quality_weight * quality_score
    )
    
    return round(hybrid_score, 4)

# Compute hybrid retrieval score
processed["retrieval_score"] = processed.apply(compute_hybrid_retrieval_score, axis=1)

# Re-sort by combined score
processed = processed.sort_values("retrieval_score", ascending=False).reset_index(drop=True)

print(f"✓ Hybrid retrieval scores computed")
print(f"\nRetrieval Score Statistics:")
print(f"  Mean: {processed['retrieval_score'].mean():.4f}")
print(f"  Std: {processed['retrieval_score'].std():.4f}")
print(f"  Min: {processed['retrieval_score'].min():.4f}")
print(f"  Max: {processed['retrieval_score'].max():.4f}")

print(f"\nTop 10 Candidates by Retrieval Score:")
display_cols = ["candidate_id", "semantic_similarity", "skill_match", "retrieval_score", "years_experience"]
print(processed[display_cols].head(10).to_string())

HYBRID RETRIEVAL SCORE COMPUTATION
✓ Hybrid retrieval scores computed

Retrieval Score Statistics:
  Mean: 0.4329
  Std: 0.0676
  Min: 0.1981
  Max: 0.7468

Top 10 Candidates by Retrieval Score:
   candidate_id  semantic_similarity  skill_match  retrieval_score  years_experience
0  CAND_0055992               0.7362     0.688784           0.7468              16.9
1  CAND_0020877               0.9235     0.822691           0.7211               5.1
2  CAND_0039754               0.4702     0.885060           0.7004              16.2
3  CAND_0095619               0.6039     0.747076           0.6980              15.6
4  CAND_0010770               0.6676     0.687615           0.6977              15.2
5  CAND_0081686               0.8648     0.769204           0.6901               6.0
6  CAND_0041669               0.7654     0.795694           0.6847               8.0
7  CAND_0079387               0.7488     0.824935           0.6745               6.9
8  CAND_0002025               0.6284    

## 6. Output & Final Results

In [7]:
print("="*80)
print("FINAL OUTPUT")
print("="*80)

# Select required columns
required_columns = [
    "candidate_id",
    "candidate_document",
    "semantic_similarity",
    "skill_match",
    "retrieval_score",
]

strongly_recommended_columns = [
    "skills",
    "career_history",
    "profile",
    "education",
    "redrob_signals",
    "years_experience",
]

output_df = processed[required_columns + strongly_recommended_columns].copy()

# Save to parquet
output_df.to_parquet(OUTPUT_PATH, index=False, engine="pyarrow")
print(f"✓ Saved {len(output_df)} candidates to {OUTPUT_PATH}")

# Save to CSV (sample)
output_df[required_columns].to_csv(CSV_OUTPUT_PATH, index=False)
print(f"✓ Saved CSV sample to {CSV_OUTPUT_PATH}")

# Generate summary report
print(f"\n" + "="*80)
print(f"PROCESSING COMPLETE - SUMMARY REPORT")
print(f"="*80)

print(f"\nOutput Statistics:")
print(f"  Total Candidates: {len(output_df)}")
print(f"  Output File: {OUTPUT_PATH}")
print(f"  File Size: {OUTPUT_PATH.stat().st_size / 1024**2:.2f} MB")

print(f"\nScore Distributions:")
print(f"  Semantic Similarity - Mean: {output_df['semantic_similarity'].mean():.4f}, Median: {output_df['semantic_similarity'].median():.4f}")
print(f"  Skill Match - Mean: {output_df['skill_match'].mean():.4f}, Median: {output_df['skill_match'].median():.4f}")
print(f"  Retrieval Score - Mean: {output_df['retrieval_score'].mean():.4f}, Median: {output_df['retrieval_score'].median():.4f}")

print(f"\nSample Output (Top 5):")
sample_output = []
for idx, row in output_df.head(5).iterrows():
    sample_output.append({
        "candidate_id": row["candidate_id"],
        "semantic_similarity": row["semantic_similarity"],
        "skill_match": row["skill_match"],
        "retrieval_score": row["retrieval_score"]
    })

import json
print(json.dumps(sample_output, indent=2))

print(f"\n✓ Pipeline execution complete!")

FINAL OUTPUT
✓ Saved 10000 candidates to top_10k_candidates.parquet
✓ Saved CSV sample to top_10k_candidates.csv

PROCESSING COMPLETE - SUMMARY REPORT

Output Statistics:
  Total Candidates: 10000
  Output File: top_10k_candidates.parquet
  File Size: 6.69 MB

Score Distributions:
  Semantic Similarity - Mean: 0.4919, Median: 0.4968
  Skill Match - Mean: 0.4328, Median: 0.4198
  Retrieval Score - Mean: 0.4329, Median: 0.4334

Sample Output (Top 5):
[
  {
    "candidate_id": "CAND_0055992",
    "semantic_similarity": 0.7361999750137329,
    "skill_match": 0.6887840017753044,
    "retrieval_score": 0.7468
  },
  {
    "candidate_id": "CAND_0020877",
    "semantic_similarity": 0.9235000014305115,
    "skill_match": 0.8226905156823304,
    "retrieval_score": 0.7211
  },
  {
    "candidate_id": "CAND_0039754",
    "semantic_similarity": 0.4702000021934509,
    "skill_match": 0.8850604144084168,
    "retrieval_score": 0.7004
  },
  {
    "candidate_id": "CAND_0095619",
    "semantic_similari